# Known-structure nanoparticle ptychography validation

By default this notebook builds the established defective Au nanoparticle. It can instead copy any ASE-readable atomic model or in-memory `ase.Atoms` object, derive plan and cross views from the same common-world atom table, and prepare independently cached validation conditions for abTEM. The prepared reference coordinates remain ground truth; frozen-phonon displacements are nuisance realizations. No CMEP source TIFF is read or modified here.


In [ ]:
import sys

# Repair output streams if an earlier Windows session repeatedly initialized
# Colorama. This is idempotent and preserves Jupyter's underlying OutStream.
def _unwrap_colorama_stream(stream):
    seen = set()
    while type(stream).__module__.startswith('colorama.'):
        if id(stream) in seen:
            break
        seen.add(id(stream))
        wrapped = getattr(stream, '_StreamWrapper__wrapped', None)
        if wrapped is None:
            break
        stream = wrapped
    return stream

sys.stdout = _unwrap_colorama_stream(sys.stdout)
sys.stderr = _unwrap_colorama_stream(sys.stderr)

from pathlib import Path
import shutil
import numpy as np

# Locate the project before importing its local modules. This supports launching
# Jupyter from this folder, its parent, or a child output folder.
working_dir = Path.cwd().resolve()
project_candidates = (working_dir, working_dir / 'New-CMEP-Method', *working_dir.parents)
project_dir = next(
    (candidate for candidate in project_candidates
     if (candidate / 'cmep_au_model.py').is_file()
     and (candidate / 'cmep_abtem_simulation.py').is_file()),
    None,
)
if project_dir is None:
    raise RuntimeError(
        'Could not locate New-CMEP-Method. Launch Jupyter from that folder or its parent.'
    )
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

# Import project modules normally. Restart the kernel after editing a local
# .py file; broad autoreload is unsafe with tqdm/colorama on Windows.
from cmep_au_model import (
    export_ground_truth, export_oriented_view, make_atomic_model_figure, make_view_frame,
    orient_atoms_for_abtem, prepare_atomic_model, print_model_summary,
    view_frame_fingerprint,
)
from cmep_abtem_simulation import (
    PtychographyConfig, analyze_4dstem_quality, configure_abtem_runtime,
    estimate_simulation_resources, export_oracle_potential, load_4dstem,
    make_validation_conditions, plot_4dstem_quality,
    print_4dstem_quality_summary, print_environment_report,
    print_resource_estimate, print_validation_conditions,
    reconstruct_multislice_ptychography, simulate_4dstem,
)


## Configuration

The 4 nm and 8 nm cases use the same structure recipe and simulation settings; only `particle_diameter_nm` changes. Beam vectors may be integer zone axes or arbitrary floating vectors in the common world frame. The projection label records image rows first and columns second.


In [ ]:
particle_diameter_nm = 4.0
# Use 4 nm as the smaller end-to-end smoke particle and 8 nm as the realistic case.
# Numerical and physical settings remain identical between sizes, so 4 nm is still
# a substantial simulation. Review the resource report before enabling 4D-STEM.

# None keeps the default Au generator. Alternatively supply an ASE-readable
# Path/string or an already loaded ase.Atoms object. Source objects/files are copied.
input_model = None
particle_shape = 'marks_decahedron'  # Used only by the default Au generator.
lattice_constant_angstrom = 4.078  # Au generator and default Au vacancy cutoff.

# Model preparation is explicit. 'preserve' retains input PBC in the truth model,
# although each abTEM beam view below is intentionally a finite nonperiodic cell.
recenter_model = True
pbc_policy = 'disable'  # 'disable', 'preserve', or 'require_nonperiodic'
defect_mode = 'vacancy_cluster'  # Use 'none' to preserve an external model.
vacancy_fraction = 0.002
vacancy_center_fraction = (0.18, -0.12, 0.08)
# Required when adding vacancies to an external model; None uses the Au default
# only for the generated particle. This is the near-vacancy labelling cutoff.
vacancy_neighbor_cutoff_angstrom = None

# Interactive known-structure viewer settings. These affect display only.
truth_viewer_background_color = 'black'
truth_viewer_marker_size = 4.0
truth_viewer_near_vacancy_marker_size = 7.0
truth_viewer_atom_opacity = 0.85
truth_viewer_figure_size = 850

# These are Cartesian vectors in the model's common-world frame, not automatically
# crystallographic Miller indices. An (h,k,l) tuple is a true zone axis only when
# the model's crystallographic basis is known to align with world x/y/z.
# Projection labels describe the zero-roll reference orientation; the exact stored
# row/column/beam vectors remain authoritative after an in-plane rotation.
plan_projection_label = 'yx'
plan_zone_axis = (0.0, 0.0, 1.0)
plan_image_up = (0.0, 1.0, 0.0)
plan_in_plane_rotation_deg = 0.0

cross_projection_label = 'zx'
cross_zone_axis = (0.0, -1.0, 0.0)
cross_image_up = (0.0, 0.0, 1.0)
cross_in_plane_rotation_deg = 0.0
vacuum_angstrom = 6.0

compute_device = 'gpu'  # 'cpu' or 'gpu'
# A scalar thermal sigma applies to every element. For mixed compositions, use
# a complete mapping such as {'Au': 0.08, 'Ag': 0.09}; values are in angstrom.
thermal_sigma_angstrom = 0.08

base_config = PtychographyConfig(
    condition_name='instrument_model',
    device=compute_device,
    energy_ev=200_000.0,
    semiangle_mrad=25.0,
    probe_aperture_soft=True,
    potential_sampling_angstrom=0.20,
    potential_slice_thickness_angstrom=1.0,
    potential_parametrization='lobato',
    potential_projection='finite',
    scan_step_angstrom=0.50,
    scan_margin_angstrom=3.0,
    detector_max_angle_mrad=50.0,
    frozen_phonon_configs=4,
    thermal_sigma_angstrom=thermal_sigma_angstrom,  # Calibrate to temperature/Debye-Waller data.
    dose_electrons_per_angstrom2=100_000.0,
    reconstruction_slice_thickness_angstrom=2.0,
    reconstruction_iterations=20,
    random_seed=17,  # Master seed; each physical view gets a deterministic namespace.
    max_batch='auto',
    cpu_chunk_size='128 MB',
    gpu_chunk_size='512 MB',
    # Instrument values remain disabled until measured or deliberately chosen.
    probe_aberrations=None,  # C lengths in A; phi angles in radians.
    beam_tilt_mrad=(0.0, 0.0),
    # This is abTEM's spatial Gaussian source-size model, not temporal coherence.
    partial_coherence_source_sigma_angstrom=0.0,
    # Independent Gaussian errors per scan component; correlated drift is not modeled.
    scan_position_error_std_angstrom=0.0,
    detector_background_mean_counts=0.0,
    detector_read_noise_std_counts=0.0,
    detector_gain_std_fraction=0.0,
    detector_dead_pixel_fraction=0.0,
    detector_saturation_counts=None,
)
validation_conditions = make_validation_conditions(base_config)

# MS-PIE optimizer controls. These affect reconstruction only, so cached
# 4D-STEM simulations can be reused when these values change.
object_step_size = 0.25
probe_step_size = 0.05
step_size_damping_rate = 0.995
probe_correction_start_iteration = 3  # 0 starts immediately; None disables it.
position_correction = False  # Exact simulated positions normally need no correction.
# Use a distinct label when comparing optimizer settings without overwriting results.
reconstruction_run_label = 'stabilized'

# Select any subset of: ideal_static, thermal_only, thermal_dose_limited,
# instrument_model. Each condition and view has its own cache.
selected_simulation_conditions = ('ideal_static',)
selected_reconstruction_conditions = selected_simulation_conditions
# Use ('plan',) for the cheapest infrastructure check; use both for CMEP.
selected_simulation_views = ('plan',)
# selected_simulation_views = ('plan', 'cross')
selected_reconstruction_views = selected_simulation_views

build_oracle_slices = False
run_4dstem_simulation = False  # Review resource estimates before enabling.
run_4dstem_qc = True  # Runs only for newly simulated or compatible cached data.
run_multislice_reconstruction = True
require_qc_pass_before_reconstruction = True

# Keep overwrite decisions stage-specific. Enabling one cannot silently recompute
# an expensive cache from another stage.
overwrite_model_exports = False
overwrite_oracle_outputs = False
overwrite_4dstem_outputs = False
overwrite_qc_outputs = False
overwrite_reconstruction_outputs = False

qc_histogram_bins = 80
qc_colormap = 'magma'

unknown_conditions = set(selected_simulation_conditions) - set(validation_conditions)
if unknown_conditions:
    raise ValueError(f'Unknown selected conditions: {sorted(unknown_conditions)}')
unknown_views = set(selected_simulation_views) - {'plan', 'cross'}
if unknown_views:
    raise ValueError(f'Unknown selected views: {sorted(unknown_views)}')

workflow_revision = 'generic_models_v1'
model_run_label = (
    f'au_{particle_diameter_nm:g}nm_{particle_shape}'
    if input_model is None else 'external_model'
)
run_tag = f'{model_run_label}_{workflow_revision}'
run_dir = project_dir / 'atomic_model_validation_outputs' / run_tag
model_dir = run_dir / 'known_structure'
view_dir = run_dir / 'oriented_views'
oracle_dir = run_dir / 'oracle_potentials'
data_dir = run_dir / '4dstem'
qc_dir = run_dir / 'quality_control'
reconstruction_dir = run_dir / 'reconstructions'
cache_dir = run_dir / '.runtime_cache'
for directory in (
    model_dir, view_dir, oracle_dir, data_dir, qc_dir,
    reconstruction_dir, cache_dir,
):
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
runtime = configure_abtem_runtime(base_config, cache_dir=cache_dir)

# Import pyplot only after configure_abtem_runtime sets the project-local Matplotlib cache. This avoids user-profile cache and permission surprises.
import matplotlib.pyplot as plt

print_environment_report(runtime)
print_validation_conditions(validation_conditions)

## Known structure

The default pre-vacancy Au Marks decahedron is called the parent, not pristine, because it already contains twin boundaries. For an external input, the parent is the copied structure before the explicitly selected workflow modification. Use `defect_mode='none'` to preserve that structure. The final prepared atom table is the single reference truth used by both beam views. This workflow does **not** relax coordinates; supply a separately relaxed known model before treating surface or defect geometry as physically equilibrated.


In [ ]:
atomic_model = prepare_atomic_model(
    particle_diameter_nm,
    input_model=input_model,
    particle_shape=particle_shape,
    lattice_constant_angstrom=lattice_constant_angstrom,
    recenter_model=recenter_model,
    pbc_policy=pbc_policy,
    defect_mode=defect_mode,
    vacancy_fraction=vacancy_fraction,
    vacancy_center_fraction=vacancy_center_fraction,
    vacancy_neighbor_cutoff_angstrom=vacancy_neighbor_cutoff_angstrom,
)
truth_paths = export_ground_truth(
    atomic_model, model_dir, run_tag=run_tag, overwrite=overwrite_model_exports
)
print_model_summary(atomic_model)
truth_paths


In [ ]:
truth_figure = make_atomic_model_figure(
    atomic_model,
    background_color=truth_viewer_background_color,
    marker_size=truth_viewer_marker_size,
    near_vacancy_marker_size=truth_viewer_near_vacancy_marker_size,
    atom_opacity=truth_viewer_atom_opacity,
    figure_size=truth_viewer_figure_size,
)
truth_figure.show(config={'scrollZoom': True, 'displaylogo': False})


## Physical beam views

`zone_axis`, `image_up`, and an independent in-plane angle define each full orthonormal frame. They are Cartesian vectors in the known model frame. The manifest stores requested vectors, normalized image-row/image-column/beam vectors, the zero-roll reference label, rotation angle, and both world/view matrices.


In [ ]:
frames = {
    'plan': make_view_frame(
        'plan', plan_zone_axis, plan_image_up,
        in_plane_rotation_deg=plan_in_plane_rotation_deg,
        projection_label=plan_projection_label,
    ),
    'cross': make_view_frame(
        'cross', cross_zone_axis, cross_image_up,
        in_plane_rotation_deg=cross_in_plane_rotation_deg,
        projection_label=cross_projection_label,
    ),
}
view_tags = {
    name: f'{name}_{frame.projection_label}_{view_frame_fingerprint(frame)[:10]}'
    for name, frame in frames.items()
}
views = {
    name: orient_atoms_for_abtem(atomic_model.atoms, frame, vacuum_angstrom=vacuum_angstrom)
    for name, frame in frames.items()
}
view_paths = {
    name: export_oriented_view(
        view, view_dir, run_tag=f'{run_tag}_{view_tags[name]}',
        model_fingerprint=atomic_model.metadata['model_fingerprint_sha256'],
        overwrite=overwrite_model_exports,
    )
    for name, view in views.items()
}
for name, view in views.items():
    print(f'{name} ({view.frame.projection_label} zero-roll reference)')
    print('  row world:', np.round(view.frame.image_up, 6))
    print('  col world:', np.round(view.frame.image_right, 6))
    print('  beam world:', np.round(view.frame.zone_axis, 6))
    print('  in-plane rotation:', view.frame.in_plane_rotation_deg, 'deg')


In [ ]:
resource_estimates = {
    (condition, name): estimate_simulation_resources(
        views[name], validation_conditions[condition]
    )
    for condition in selected_simulation_conditions
    for name in selected_simulation_views
}
for (condition, name), estimate in resource_estimates.items():
    print_resource_estimate(estimate, label=f'{condition}/{name}')
    print()

total_raw_gib = sum(
    estimate.details['stored_4dstem_float32_gib']
    for estimate in resource_estimates.values()
)
largest_unreduced_gib = max(
    (estimate.details['unreduced_frozen_phonon_float32_gib']
     for estimate in resource_estimates.values()),
    default=0.0,
)
disk_free_gib = shutil.disk_usage(run_dir).free / 1024**3
print(f'Selected-condition raw output total: ~{total_raw_gib:.2f} GiB')
print(f'Largest unreduced frozen-phonon equivalent: ~{largest_unreduced_gib:.2f} GiB')
print(f'Free disk at output location: {disk_free_gib:.2f} GiB')
if total_raw_gib * 1.5 > disk_free_gib:
    print('WARNING: free disk is less than 1.5x the estimated raw output total.')


## Static oracle potential slices

These are noise-free simulator truth products, not ptychographic reconstructions. abTEM native `(slice,x,y)` arrays are exported as `(slice,row=y,col=x)` inside each physical view frame.


In [ ]:
oracle_results = {}
if build_oracle_slices:
    for name in selected_simulation_views:
        view = views[name]
        oracle_results[name] = export_oracle_potential(
            view, base_config,
            oracle_dir / f'{run_tag}_{view_tags[name]}_oracle.npz',
            model_fingerprint=atomic_model.metadata['model_fingerprint_sha256'],
            overwrite=overwrite_oracle_outputs,
        )
        print(name, oracle_results[name].stack_slice_row_col.shape,
              oracle_results[name].output_path)
else:
    print('Oracle-potential build disabled.')


In [ ]:
if oracle_results:
    fig, axes = plt.subplots(1, len(oracle_results), figsize=(12, 5), constrained_layout=True, dpi=200)
    axes = np.atleast_1d(axes)
    for ax, (name, result) in zip(axes, oracle_results.items()):
        index = len(result.z_angstrom) // 2
        dx = float(np.median(np.diff(result.x_angstrom)))
        dy = float(np.median(np.diff(result.y_angstrom)))
        extent_nm = [
            (result.x_angstrom[0] - dx / 2) / 10,
            (result.x_angstrom[-1] + dx / 2) / 10,
            (result.y_angstrom[0] - dy / 2) / 10,
            (result.y_angstrom[-1] + dy / 2) / 10,
        ]
        image = ax.imshow(
            result.stack_slice_row_col[index], origin='lower', cmap='magma',
            extent=extent_nm,
        )
        ax.set(title=f'{name}: oracle slice {index}',
               xlabel='view x (nm)', ylabel='view y (nm)')
        fig.colorbar(image, ax=ax, shrink=0.75)
    plt.show()


## Independently cached 4D-STEM conditions

Enable simulation only after reviewing the per-view and aggregate estimates. There is no atom, scan-position, detector-pixel, visualization, or output-size cap. The simulation-state sidecar stores nominal/actual scan coordinates and any realized fixed detector gain/dead-pixel maps. Cache leaf names are intentionally compact because the enclosing run and condition directories already identify them and Zarr needs extra path headroom for atomic writes on Windows. A partial Zarr without a manifest is rejected; rerun that stage with its specific overwrite switch after inspecting the failed output.


In [ ]:
simulation_results = {}
for condition in selected_simulation_conditions:
    config = validation_conditions[condition]
    condition_dir = data_dir / condition
    condition_dir.mkdir(parents=True, exist_ok=True)
    for name in selected_simulation_views:
        view = views[name]
        key = (condition, name)
        data_path = condition_dir / f'{view_tags[name]}_4dstem.zarr'
        if run_4dstem_simulation:
            simulation_results[key] = simulate_4dstem(
                view, config, data_path,
                model_fingerprint=atomic_model.metadata['model_fingerprint_sha256'],
                overwrite=overwrite_4dstem_outputs,
            )
        elif data_path.exists() and data_path.with_suffix('.json').exists():
            cached = load_4dstem(data_path)
            compatible = (
                cached.metadata.get('configuration') == config.to_dict()
                and cached.metadata.get('view') == view.metadata
                and cached.metadata.get('model_fingerprint_sha256')
                    == atomic_model.metadata['model_fingerprint_sha256']
            )
            if not compatible:
                raise RuntimeError(f'Cached settings do not match {condition}/{name}: {data_path}')
            simulation_results[key] = cached

if simulation_results:
    for key, result in simulation_results.items():
        print(key, result.diffraction_patterns.shape, result.output_path)
else:
    print('4D-STEM simulation disabled and no selected compatible cache was loaded.')


## Quality control before reconstruction

QC reads the complete lazy dataset without subsampling and materializes only compact diagnostics: the mean diffraction pattern in mrad, scan-integrated image in physical scan coordinates, scan-position signal histogram, finite/NaN/inf/zero checks, exact dimensions, and actual Zarr size.


In [ ]:
quality_control_results = {}
if run_4dstem_qc and simulation_results:
    for (condition, name), simulation in simulation_results.items():
        condition_qc_dir = qc_dir / condition
        condition_qc_dir.mkdir(parents=True, exist_ok=True)
        result = analyze_4dstem_quality(
            simulation,
            condition_qc_dir / f'{view_tags[name]}_qc.npz',
            histogram_bins=qc_histogram_bins,
            overwrite=overwrite_qc_outputs,
        )
        quality_control_results[(condition, name)] = result
        print_4dstem_quality_summary(result, label=f'{condition}/{name}')
        plot_4dstem_quality(result, cmap=qc_colormap)
        plt.show()
elif run_4dstem_qc:
    print('QC is enabled, but no selected simulation data is available.')
else:
    print('4D-STEM QC disabled.')


## Optional multislice ptychographic reconstruction

The current abTEM MS-PIE operator is in-memory. A realistic reconstruction may require substantially more RAM or VRAM than the compressed Zarr size. By default, reconstruction is blocked until the complete selected dataset passes finite-value and nonzero-signal QC. Simulated scan errors remain unknown to reconstruction, which intentionally receives the nominal raster.


In [ ]:
reconstruction_results = {}
if run_multislice_reconstruction:
    for condition in selected_reconstruction_conditions:
        if condition not in validation_conditions:
            raise ValueError(f'Unknown reconstruction condition: {condition}')
        for name in selected_reconstruction_views:
            view = views[name]
            key = (condition, name)
            if key not in simulation_results:
                raise RuntimeError(f'Missing 4D-STEM data for {condition}/{name}')
            if require_qc_pass_before_reconstruction:
                if key not in quality_control_results:
                    raise RuntimeError(f'Missing required QC result for {condition}/{name}')
                qc_meta = quality_control_results[key].metadata
                qc_passed = (
                    qc_meta['finite_value_count'] == qc_meta['total_value_count']
                    and qc_meta['nan_value_count'] == 0
                    and qc_meta['infinite_value_count'] == 0
                    and qc_meta['maximum'] > 0.0
                )
                if not qc_passed:
                    raise RuntimeError(f'QC failed for {condition}/{name}; reconstruction blocked.')
            condition_reconstruction_dir = reconstruction_dir / condition
            condition_reconstruction_dir.mkdir(parents=True, exist_ok=True)
            result = reconstruct_multislice_ptychography(
                simulation_results[key], view, validation_conditions[condition],
                condition_reconstruction_dir
                    / f'{view_tags[name]}_{reconstruction_run_label}_mspie.npz',
                object_step_size=object_step_size,
                probe_step_size=probe_step_size,
                step_size_damping_rate=step_size_damping_rate,
                probe_correction_start_iteration=probe_correction_start_iteration,
                position_correction=position_correction,
                overwrite=overwrite_reconstruction_outputs,
            )
            reconstruction_results[key] = result
            print(key, result.phase_stack_slice_row_col.shape,
                  'error=', result.error, result.output_path)
else:
    print('Multislice reconstruction disabled.')


## Validation handoff

The next stage will pass the two reconstructed phase stacks through the generalized CMEP workflow, map localized estimates back through each saved view transform, and compare them with the surviving reference `atom_id` table. Metrics can be stratified across bulk, vacancy-neighbour, and twin-boundary atoms after a separate structural-classification step derives trustworthy boundary labels.
